# DesertMap EDA

This notebook explores the USDA Food Access Research Atlas 2019 tract-level dataset for the DesertMap portfolio project. The target throughout this notebook is `LILATracts_1And10`, the USDA low-income and low-access standard using the 1-mile urban and 10-mile rural threshold.

## Section 0: Setup

This section imports the core analysis libraries, configures plotting defaults, and loads the USDA Food Access Research Atlas data. It also creates the output directory used for exported figures.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data/raw/FoodAccessResearchAtlasData2019.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'LILATracts_1And10'
DOWNLOAD_URL = 'https://www.ers.usda.gov/data-products/food-access-research-atlas/'

if not DATA_PATH.exists():
    print(f'ERROR: Expected USDA Food Access Research Atlas CSV not found at {DATA_PATH}.')
    print(f'Download the data from: {DOWNLOAD_URL}')
    raise FileNotFoundError(f'Missing required data file: {DATA_PATH}')

food_access = pd.read_csv(DATA_PATH, low_memory=False)

print(f'Dataset shape: {food_access.shape[0]:,} rows x {food_access.shape[1]:,} columns')
food_access.head()

OSError: [Errno 30] Read-only file system: '/outputs'

## Section 1: Target Variable — LILATracts_1And10

This section examines the food desert target definition used for modeling. Understanding class balance is important because imbalanced labels affect model evaluation and threshold choices.

In [ ]:
if TARGET_COL not in food_access.columns:
    raise KeyError(f'Required target column not found: {TARGET_COL}')

target_counts = food_access[TARGET_COL].value_counts(dropna=False).sort_index()
target_percentages = food_access[TARGET_COL].value_counts(normalize=True, dropna=False).sort_index() * 100

target_summary = pd.DataFrame({
    'count': target_counts,
    'percentage': target_percentages.round(2)
})

print('Target value counts and percentages:')
display(target_summary)

food_desert_rate = food_access[TARGET_COL].mean() * 100
print(f'{food_desert_rate:.1f}% of U.S. census tracts qualify as food deserts under the USDA standard definition.')

In [ ]:
target_plot_data = (
    food_access[TARGET_COL]
    .map({0: 'Not food desert', 1: 'Food desert'})
    .value_counts()
    .reindex(['Not food desert', 'Food desert'])
    .reset_index()
)
target_plot_data.columns = ['Food desert status', 'Number of census tracts']

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=target_plot_data,
    x='Food desert status',
    y='Number of census tracts',
    hue='Food desert status',
    palette=['#4C78A8', '#F58518'],
    legend=False,
    ax=ax
)
ax.set_title('USDA Food Desert Class Distribution')
ax.set_xlabel('Food desert status')
ax.set_ylabel('Number of census tracts')
plt.tight_layout()
fig.savefig(FIGURE_DIR / "01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 2: Feature Inventory

This section organizes available columns into practical categories and defines the initial modeling feature list. It also checks missingness for the features and target used in later notebooks.

In [ ]:
all_columns = list(food_access.columns)

identity_columns = [col for col in ['CensusTract', 'TractFIPS', 'FIPS', 'State', 'County'] if col in food_access.columns]

demographic_keywords = [
    'pct_', 'poverty', 'income', 'snap', 'hunv', 'vehicle', 'pop', 'white', 'black',
    'hisp', 'asian', 'aian', 'nhopi', 'kids', 'seniors', 'lowi'
]
demographic_columns = [
    col for col in all_columns
    if any(keyword in col.lower() for keyword in demographic_keywords)
]

distance_keywords = ['lapop', 'lalowi', 'lakids', 'laseniors', 'lahunv', 'lasnap', 'lawhite', 'lablack', 'laasian', 'lanhopi', 'laaian', 'laomultir', 'lahisp']
food_access_distance_columns = [
    col for col in all_columns
    if any(col.lower().startswith(keyword) for keyword in distance_keywords)
]

food_desert_flag_columns = [
    col for col in all_columns
    if col.startswith('LILA') or col.startswith('LA') or col.startswith('LATracts')
]

column_groups = {
    'Identity columns': identity_columns,
    'Demographic columns': demographic_columns,
    'Food access distance columns': food_access_distance_columns,
    'Food desert flag columns': food_desert_flag_columns,
}

for group_name, columns in column_groups.items():
    print(f'\n{group_name} ({len(columns)}):')
    for column in columns:
        print(f'  - {column}')

In [ ]:
MODEL_FEATURES = [
    'MedianFamilyIncome',
    'PovertyRate',
    'pct_nhblack10',
    'pct_hisp10',
    'pct_nhwhite10',
    'pct_hunv10',
    'pct_snap16',
    'Pop2010',
    'Urban',
]

required_columns = MODEL_FEATURES + [TARGET_COL]
missing_required_columns = [col for col in required_columns if col not in food_access.columns]

if missing_required_columns:
    raise KeyError(f'Missing required columns: {missing_required_columns}')

print('Final MODEL_FEATURES list:')
for feature in MODEL_FEATURES:
    print(f'  - {feature}')

print('\nMissing value counts for MODEL_FEATURES and target:')
display(food_access[required_columns].isna().sum().to_frame('missing_count'))

## Section 3: Distribution Plots

This section compares key feature distributions by food desert status. These plots help identify separation between classes, skewed variables, and potential preprocessing needs.

In [ ]:
distribution_features = [
    'MedianFamilyIncome',
    'PovertyRate',
    'pct_nhblack10',
    'pct_hisp10',
    'pct_hunv10',
    'pct_snap16',
]

feature_labels = {
    'MedianFamilyIncome': 'Median family income',
    'PovertyRate': 'Poverty rate (%)',
    'pct_nhblack10': 'Non-Hispanic Black population (%)',
    'pct_hisp10': 'Hispanic population (%)',
    'pct_hunv10': 'Households with no vehicle (%)',
    'pct_snap16': 'SNAP households (%)',
}

plot_data = food_access[distribution_features + [TARGET_COL]].copy()
plot_data['Food desert status'] = plot_data[TARGET_COL].map({0: 'Not food desert', 1: 'Food desert'})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for ax, feature in zip(axes, distribution_features):
    sns.histplot(
        data=plot_data,
        x=feature,
        hue='Food desert status',
        kde=True,
        stat='density',
        common_norm=False,
        bins=40,
        palette=['#4C78A8', '#F58518'],
        alpha=0.35,
        ax=ax
    )
    ax.set_title(f'Distribution of {feature_labels[feature]}')
    ax.set_xlabel(feature_labels[feature])
    ax.set_ylabel('Density')

fig.suptitle('Feature Distributions by USDA Food Desert Status', fontsize=16, y=1.02)
plt.tight_layout()
fig.savefig(FIGURE_DIR / '01_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: Correlation Matrix

This section measures linear relationships among the selected features and the food desert target. Correlations are not causal, but they provide a useful first pass at which variables move with the target.

In [ ]:
correlation_columns = MODEL_FEATURES + [TARGET_COL]
correlation_matrix = food_access[correlation_columns].corr(method='pearson')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'},
    ax=ax
)
ax.set_title('Correlation Matrix for Modeling Features and Food Desert Target')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(FIGURE_DIR / '01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
target_correlations = correlation_matrix[TARGET_COL].drop(TARGET_COL).sort_values()
top_positive_correlations = target_correlations.sort_values(ascending=False).head(3)
top_negative_correlations = target_correlations.head(3)

positive_text = ', '.join(
    f'`{feature}` ({correlation:.2f})'
    for feature, correlation in top_positive_correlations.items()
)
negative_text = ', '.join(
    f'`{feature}` ({correlation:.2f})'
    for feature, correlation in top_negative_correlations.items()
)

display(Markdown(
    f'**Correlation highlights:** The three features most positively correlated with `{TARGET_COL}` are '
    f'{positive_text}. The three features most negatively correlated with `{TARGET_COL}` are {negative_text}.'
))

## Section 5: Demographic Quartile Preview

This section previews the Week 6 fairness audit by comparing food desert rates across demographic quartiles. The grouping uses continuous percentage columns divided into quartiles, which preserves more information than categorical majority-group labels.

In [ ]:
quartile_labels = ['Q1', 'Q2', 'Q3', 'Q4']
quartile_data = food_access[[TARGET_COL, 'pct_nhblack10', 'pct_hisp10']].copy()

quartile_data['Black population quartile'] = pd.qcut(
    quartile_data['pct_nhblack10'],
    q=4,
    labels=quartile_labels,
    duplicates='drop'
)
quartile_data['Hispanic population quartile'] = pd.qcut(
    quartile_data['pct_hisp10'],
    q=4,
    labels=quartile_labels,
    duplicates='drop'
)

black_quartile_rates = (
    quartile_data.groupby('Black population quartile', observed=True)[TARGET_COL]
    .mean()
    .reset_index(name='Food desert rate')
)
hispanic_quartile_rates = (
    quartile_data.groupby('Hispanic population quartile', observed=True)[TARGET_COL]
    .mean()
    .reset_index(name='Food desert rate')
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

sns.barplot(
    data=black_quartile_rates,
    x='Black population quartile',
    y='Food desert rate',
    color='#4C78A8',
    ax=axes[0]
)
axes[0].set_title('Food Desert Rate by % Black Quartile')
axes[0].set_xlabel('Non-Hispanic Black population quartile')
axes[0].set_ylabel('Food desert rate')
axes[0].set_ylim(0, max(black_quartile_rates['Food desert rate'].max(), hispanic_quartile_rates['Food desert rate'].max()) * 1.15)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f'{value:.0%}'))

sns.barplot(
    data=hispanic_quartile_rates,
    x='Hispanic population quartile',
    y='Food desert rate',
    color='#F58518',
    ax=axes[1]
)
axes[1].set_title('Food Desert Rate by % Hispanic Quartile')
axes[1].set_xlabel('Hispanic population quartile')
axes[1].set_ylabel('Food desert rate')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f'{value:.0%}'))

fig.suptitle('Week 6 Fairness Audit Preview: Food Desert Rates by Demographic Quartile', fontsize=15, y=1.03)
plt.tight_layout()
fig.savefig(FIGURE_DIR / '01_demographic_quartile_preview.png', dpi=150, bbox_inches='tight')
plt.show()

display(black_quartile_rates)
display(hispanic_quartile_rates)

## Section 6: Summary and Next Steps

This section summarizes the dataset, selected feature set, class balance, and strongest target correlations. It closes with concrete preparation tasks for `02_cleaning.ipynb`.

In [ ]:
summary_table = pd.DataFrame({
    'Metric': [
        'Dataset size',
        'Model feature count',
        'Food desert class balance',
        'Top positive target correlations',
        'Top negative target correlations',
    ],
    'Value': [
        f'{food_access.shape[0]:,} rows x {food_access.shape[1]:,} columns',
        f'{len(MODEL_FEATURES)} features',
        f'{food_desert_rate:.1f}% food desert / {100 - food_desert_rate:.1f}% not food desert',
        ', '.join(f'{feature} ({correlation:.2f})' for feature, correlation in top_positive_correlations.items()),
        ', '.join(f'{feature} ({correlation:.2f})' for feature, correlation in top_negative_correlations.items()),
    ]
})

summary_markdown = '| Metric | Value |\n|---|---|\n'
summary_markdown += '\n'.join(
    f'| {row.Metric} | {row.Value} |'
    for row in summary_table.itertuples(index=False)
)

display(Markdown(summary_markdown))

### Next Steps for `02_cleaning.ipynb`

- Define a missing value strategy for each modeling feature, including whether to impute, drop, or add missingness indicators.
- Review skew and extreme values in income, poverty, vehicle access, SNAP rate, and population count before choosing outlier handling rules.
- Construct the final feature matrix with `MODEL_FEATURES`, verify the target remains `LILATracts_1And10`, and save a clean modeling dataset for training.